# Unified Extrapolation Model (UEM) Example

This notebook demonstrates the UEM extrapolation method as an alternative to Muggianu for multicomponent thermodynamics.

**Reference**: Ju, T., et al. (2024) *Thermochimica Acta* 740, 179824

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pycalphad import Database, Model, calculate, variables as v
from pycalphad.models.model_uem import ModelUEM

## Load Database

We'll use the Al-Cr-Ni database as an example.

In [ ]:
dbf = Database('alcrni.tdb')
comps = ['AL', 'CR', 'NI', 'VA']
phases = ['LIQUID']

## Binary System: UEM vs Standard

For binary systems, UEM should give identical results to standard Redlich-Kister.

In [ ]:
# Binary Al-Ni
x_al = np.linspace(0.01, 0.99, 50)
gm_std_binary = []
gm_uem_binary = []

for x in x_al:
    res_std = calculate(dbf, ['AL', 'NI', 'VA'], phases, T=1800, P=101325, X_AL=x, N=1)
    res_uem = calculate(dbf, ['AL', 'NI', 'VA'], phases, model=ModelUEM, T=1800, P=101325, X_AL=x, N=1)
    gm_std_binary.append(res_std.GM.values[0])
    gm_uem_binary.append(res_uem.GM.values[0])

plt.figure(figsize=(10, 6))
plt.plot(x_al, gm_std_binary, 'b-', label='Standard (Redlich-Kister)', linewidth=2)
plt.plot(x_al, gm_uem_binary, 'r--', label='UEM', linewidth=2, alpha=0.7)
plt.xlabel('X(AL)')
plt.ylabel('G_m (J/mol)')
plt.title('Binary Al-Ni: UEM vs Standard (Should be Identical)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Max difference: {np.max(np.abs(np.array(gm_std_binary) - np.array(gm_uem_binary))):.2e} J/mol")

## Ternary System: UEM vs Muggianu

For ternary systems, UEM and Muggianu give different predictions. This is expected and shows the different extrapolation methods.

In [ ]:
# Ternary at fixed temperature
T = 1800
compositions = [
    (0.33, 0.33, "Equimolar"),
    (0.50, 0.25, "AL-rich"),
    (0.25, 0.50, "CR-rich"),
    (0.25, 0.25, "NI-rich")
]

results = []
for x_al, x_cr, label in compositions:
    res_std = calculate(dbf, comps, phases, T=T, P=101325, X_AL=x_al, X_CR=x_cr, N=1)
    res_uem = calculate(dbf, comps, phases, model=ModelUEM, T=T, P=101325, X_AL=x_al, X_CR=x_cr, N=1)
    
    gm_std = res_std.GM.values[0]
    gm_uem = res_uem.GM.values[0]
    diff = gm_uem - gm_std
    pct = 100 * diff / gm_std
    
    results.append({
        'label': label,
        'X_AL': x_al,
        'X_CR': x_cr,
        'GM_Muggianu': gm_std,
        'GM_UEM': gm_uem,
        'Difference': diff,
        'Percent': pct
    })

# Display results
import pandas as pd
df = pd.DataFrame(results)
print("\nTernary Al-Cr-Ni at T=1800K:")
print(df.to_string(index=False))

## Temperature Dependence

Compare UEM and Muggianu across temperature range.

In [ ]:
# Fixed composition, vary temperature
temps = np.linspace(1000, 2000, 20)
gm_std_T = []
gm_uem_T = []

for T in temps:
    res_std = calculate(dbf, comps, phases, T=T, P=101325, X_AL=0.4, X_CR=0.3, N=1)
    res_uem = calculate(dbf, comps, phases, model=ModelUEM, T=T, P=101325, X_AL=0.4, X_CR=0.3, N=1)
    gm_std_T.append(res_std.GM.values[0])
    gm_uem_T.append(res_uem.GM.values[0])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Gibbs energy vs T
ax1.plot(temps, gm_std_T, 'b-', label='Muggianu', linewidth=2)
ax1.plot(temps, gm_uem_T, 'r--', label='UEM', linewidth=2)
ax1.set_xlabel('Temperature (K)')
ax1.set_ylabel('G_m (J/mol)')
ax1.set_title('Gibbs Energy vs Temperature\nX(AL)=0.4, X(CR)=0.3')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Difference vs T
diff = np.array(gm_uem_T) - np.array(gm_std_T)
ax2.plot(temps, diff, 'g-', linewidth=2)
ax2.axhline(y=0, color='k', linestyle='--', alpha=0.3)
ax2.set_xlabel('Temperature (K)')
ax2.set_ylabel('G_UEM - G_Muggianu (J/mol)')
ax2.set_title('Difference: UEM vs Muggianu')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Interpretation

**Binary systems**: UEM and standard models give identical results because no multicomponent extrapolation is needed.

**Ternary+ systems**: UEM and Muggianu differ because they use different extrapolation methods:
- **Muggianu**: Geometric symmetric averaging
- **UEM**: Property-difference-based effective mole fractions

Differences are typically 1-15% and are **expected and correct**. Both are valid approximations; experimental data would be needed to determine which is more accurate for a specific system.

Use UEM when:
- Binary subsystems are highly asymmetric
- Components have very different properties
- Assessing extrapolation uncertainty

## References

Ju, T., Huang, Z., Ding, X., Yan, X. & Liao, C. (2024). "A Unified Extrapolation thermodynamic model for multicomponent solutions based on binary data." *Thermochimica Acta*, 740, 179824. https://doi.org/10.1016/j.tca.2024.179824